# Google ADK Live + LangSmith

This notebook shows where the LangSmith ADK Live plugin attaches to a Gemini Live voice agent. The final run cell uses the maintained backend implementation through `workshop`.

## 1. Build The Agent

The ADK agent uses Gemini Live with two tools: current time and weather.

In [ ]:
import os
import uuid

from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.agents.live_request_queue import LiveRequestQueue
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

from voice_demo.adk.agent import (
    APP_NAME,
    MODEL,
    RECV_SAMPLE_RATE,
    SEND_SAMPLE_RATE,
    USER_ID,
    _INSTRUCTIONS,
    get_time,
    get_weather,
)
from workshop import ConsoleStatus, MicStream, SpeakerStream, run_google_adk_live

load_dotenv()

PROJECT = "voice-workshop-google-adk-live"

assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY before running this notebook."
assert os.getenv("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY before running this notebook."


In [ ]:
root_agent = LlmAgent(
    name="voice_assistant",
    model=MODEL,
    instruction=_INSTRUCTIONS,
    tools=[get_time, get_weather],
)

[get_time.__name__, get_weather.__name__], MODEL


## 2. Configure Tracing

`LangSmithGoogleADKLivePlugin` is registered on the ADK `Runner`. It traces the Live event stream while the app loop handles audio playback and UI.

In [ ]:
from langsmith.integrations.google_adk_live import LangSmithGoogleADKLivePlugin

thread_id = str(uuid.uuid4())
tracing_plugin = LangSmithGoogleADKLivePlugin(
    sample_rate=RECV_SAMPLE_RATE,
    thread_id_provider=lambda: thread_id,
    project_name=PROJECT,
    tags=["workshop", "google-adk-live"],
    metadata={"model": MODEL},
)


## 3. Put It Together

The tracing plugin becomes part of the ADK framework here via `Runner(..., plugins=[tracing_plugin])`.

In [ ]:
session_service = InMemorySessionService()
adk_session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
runner = Runner(
    app_name=APP_NAME,
    agent=root_agent,
    session_service=session_service,
    plugins=[tracing_plugin],
)
queue = LiveRequestQueue()

thread_id


## 4. Run The Agent Live

This uses the repo's console mic/speaker transport and the shared ADK backend. Stop/cancel the cell to end the voice session.

In [ ]:
audio_in = MicStream(sample_rate=RECV_SAMPLE_RATE)
audio_out = SpeakerStream(sample_rate=RECV_SAMPLE_RATE)
ui = ConsoleStatus()

await run_google_adk_live(PROJECT, audio_in=audio_in, audio_out=audio_out, ui=ui)
